## Middel Ware

In [1]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
import os

In [2]:
!ollama pull deepseek-v2

pulling manifest ⠋ pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠼ pulling manifest ⠦ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠋ pulling manifest ⠋ pulling manifest ⠹ pulling manifest 
pulling d8d69f2a1bfa: 100% ▕██████████████████▏ 8.9 GB                         
pulling b321cd7de6c7: 100% ▕██████████████████▏  111 B                         
pulling 4bb71764481f: 100% ▕██████████████████▏  13 KB                         
pulling 19f2fb9e8bc6: 100% ▕██████████████████▏   32 B                         
pulling 2b2c35718a16: 100% ▕██████████████████▏  494 B                         
verifying sha256 digest 
writing manifest ⠋ pulling manifest 
pulling d8d69f2a1bfa: 100% ▕██████████████████▏ 8.9 GB                         
pulling b321cd7de6c7: 100% ▕██████████████████▏  111 B                         
pulling 4bb71764481f: 100% ▕██████████████████▏  13 KB                         
pulling 19f2fb9e8bc6: 100% ▕█████

In [3]:
model=ChatOllama(model="deepseek-v2",api_key="ollama",temperature=0.2)

### 1.Summarization Middelware

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

In [7]:
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [8]:
config={"configurable":{"thread_id":"test-1"}}

In [9]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='37a3904e-93cc-4f0b-89dc-27163ca82450'), AIMessage(content=' 2+2 equals 4.', additional_kwargs={}, response_metadata={'model': 'deepseek-v2', 'created_at': '2026-10-29T08:47:34.6507238Z', 'done': True, 'done_reason': 'stop', 'total_duration': 48403508000, 'load_duration': 40153520300, 'prompt_eval_count': 14, 'prompt_eval_duration': 5332312000, 'eval_count': 9, 'eval_duration': 2797234000, 'logprobs': None, 'model_name': 'deepseek-v2', 'model_provider': 'ollama'}, id='lc_run--01a186d7-19df-72e4-bb14-7cae730c3016-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 9, 'total_tokens': 23})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='37a3904e-93cc-4f0b-89dc-27163ca82450'), AIMessage(content=' 2+2 equals 4.', additional_kwargs={}, response_metadata={'model': 'deeps

### middelware example2

In [1]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

model=ChatOllama(model="deepseek-v2",api_key="ollama",temperature=0.2)

agent=create_agent(model=model,checkpointer=InMemorySaver(),
middleware=[SummarizationMiddleware(model=model,trigger=("messages",10),keep=("messages",4))]
)

In [2]:
config={"configurable":{"thread_id":"test1"}}

In [4]:
from langchain_core.messages import content
question=[
"What is AI?",
"What is ML?",
"What is Deep Learning?",
"What is 100/4?",
"What is 15-7?",
"What is 3*3?",
"What is 4*4?",
]
for q in question:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is AI?', additional_kwargs={}, response_metadata={}, id='b07d0471-8b6d-4e92-bc39-8ff9bc194697'), AIMessage(content=' AI, or Artificial Intelligence, refers to the development of computer systems that can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI systems are designed to learn from experience and adapt to new data, making them increasingly accurate and efficient over time. AI technologies are used in a wide range of applications, including natural language processing, image recognition, and autonomous vehicles.', additional_kwargs={}, response_metadata={'model': 'deepseek-v2', 'created_at': '2026-10-29T14:47:27.8423342Z', 'done': True, 'done_reason': 'stop', 'total_duration': 56606919200, 'load_duration': 32799609200, 'prompt_eval_count': 11, 'prompt_eval_duration': 3873267000, 'eval_count': 86, 'eval_duration': 19812878000, 'logpr

### Based On size Of token

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langchain_core.tools import tool

load_dotenv(override=True)

True

In [8]:
@tool
def Search_hotel(city:str)->str:
    """"Search hotels-return a long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


In [9]:
api_key=os.getenv("GOOGLE_API_KEY")
model=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",api_key=api_key,temperature=0.1)
agent=create_agent(model=model,tools=[Search_hotel],checkpointer=InMemorySaver(),middleware=[SummarizationMiddleware(model=model,trigger=("tokens",500),keep=("tokens",300),)])
config = {"configurable": {"thread_id": "test-1"}}

In [10]:
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 

In [11]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~202 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='46ca4f6a-0d2e-4fc0-be0a-2f4fdd9c6b67'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'Search_hotel', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_655896': 'EnMKcQFpFH0Ti3PEym9uUvX+SyUJ0SxeTUwGmsxIX/B6eKC4jdaXi/TrYrgy/nOOKdx0mi8t4E4Nek8Pfob0iYCLOdbSeyXotMMsQu4Sal76KsUwSIytfMuXzpx1PbNR0NTaj60LaKG0Jlu/AAeNDsJ211QB'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1882f-58c0-700e-97cd-359f8f025db9-0', tool_calls=[{'name': 'Search_hotel', 'args': {'city': 'Paris'}, 'id': 'call_655896', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 16, 'total_tokens': 70, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris:\n    1. G

## Fractions

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])